# Milestone 6: Grounded QA bot over course documents
*AI from scratch, milestone 6* · combine notebook for Stage 6

Build a complete retrieval-augmented generation (RAG) question-answering bot over the 254 lesson sections
of the *AI from scratch* curriculum:
1. Index course document chunks using TF-IDF and cosine vector representations.
2. Build a top-$k$ retriever and measure context recall on a 35-question human-written evaluation set.
3. Assemble grounded prompt contexts with bracketed passage citations (`[1]`, `[2]`, `[3]`).
4. Formulate grounded answers and evaluate grounding (measuring whether answer statements are faithful to retrieved facts vs hallucinations).

Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it; `hint(n)` and `answer(n)`
are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit(docs); D = tfidf.transform(docs)'], 2: ['q_vec = tfidf.transform([query]); scores = (q_vec @ D.T).toarray()[0]; return list(np.argsort(-scores)[:k])'], 3: ['hits = [target[i] in retrieve(questions[i]["q"], k=3) for i in range(len(questions))]', 'recall_at_3 = float(np.mean(hits))'], 4: ['lines = ["CONTEXT:"] + [f"[{i+1}] {docs[idx]}" for i, idx in enumerate(chunk_indices)] + [f"QUESTION: {query}", "INSTRUCTIONS: Answer using ONLY the context above. Cite sources with [1], [2]."]'], 5: ["a_words = set(re.findall(r'[a-z]{4,}', answer.lower()))", 'if not a_words: return 1.0', "c_words = set(re.findall(r'[a-z]{4,}', context_text.lower()))", 'return float(len(a_words & c_words) / len(a_words))'], 6: ['indices = retrieve(query, k=2); return {"query": query, "citations": [sections[i]["lesson"] for i in indices], "top_chunk": docs[indices[0]], "is_grounded": True}']}
_ANSWERS = {1: 'dGZpZGYgPSBUZmlkZlZlY3Rvcml6ZXIoc3RvcF93b3Jkcz0iZW5nbGlzaCIsIHN1YmxpbmVhcl90Zj1UcnVlKS5maXQoZG9jcykKRCA9IHRmaWRmLnRyYW5zZm9ybShkb2NzKQ==', 2: 'cV92ZWMgPSB0ZmlkZi50cmFuc2Zvcm0oW3F1ZXJ5XSkKc2NvcmVzID0gKHFfdmVjIEAgRC5UKS50b2FycmF5KClbMF0KcmV0dXJuIGxpc3QobnAuYXJnc29ydCgtc2NvcmVzKVs6a10p', 3: 'aGl0cyA9IFt0YXJnZXRbaV0gaW4gcmV0cmlldmUocXVlc3Rpb25zW2ldWyJxIl0sIGs9MykgZm9yIGkgaW4gcmFuZ2UobGVuKHF1ZXN0aW9ucykpXQpyZWNhbGxfYXRfMyA9IGZsb2F0KG5wLm1lYW4oaGl0cykp', 4: 'bGluZXMgPSBbIkNPTlRFWFQ6Il0KZm9yIGksIGlkeCBpbiBlbnVtZXJhdGUoY2h1bmtfaW5kaWNlcyk6CiAgICBsaW5lcy5hcHBlbmQoZiJbe2krMX1dIHtkb2NzW2lkeF19IikKbGluZXMuYXBwZW5kKGYiUVVFU1RJT046IHtxdWVyeX0iKQpsaW5lcy5hcHBlbmQoIklOU1RSVUNUSU9OUzogQW5zd2VyIHVzaW5nIE9OTFkgdGhlIGNvbnRleHQgYWJvdmUuIENpdGUgc291cmNlcyB3aXRoIFsxXSwgWzJdLiIpCnJldHVybiAiXG5cbiIuam9pbihsaW5lcyk=', 5: 'YV93b3JkcyA9IHNldChyZS5maW5kYWxsKHInW2Etel17NCx9JywgYW5zd2VyLmxvd2VyKCkpKQppZiBub3QgYV93b3JkczoKICAgIHJldHVybiAxLjAKY193b3JkcyA9IHNldChyZS5maW5kYWxsKHInW2Etel17NCx9JywgY29udGV4dF90ZXh0Lmxvd2VyKCkpKQpyZXR1cm4gZmxvYXQobGVuKGFfd29yZHMgJiBjX3dvcmRzKSAvIGxlbihhX3dvcmRzKSk=', 6: 'aW5kaWNlcyA9IHJldHJpZXZlKHF1ZXJ5LCBrPTIpCnJldHVybiB7CiAgICAicXVlcnkiOiBxdWVyeSwKICAgICJjaXRhdGlvbnMiOiBbc2VjdGlvbnNbaV1bImxlc3NvbiJdIGZvciBpIGluIGluZGljZXNdLAogICAgInRvcF9jaHVuayI6IGRvY3NbaW5kaWNlc1swXV0sCiAgICAiaXNfZ3JvdW5kZWQiOiBUcnVlCn0='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import json, urllib.request, re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

DATA = "https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/"
index = json.load(urllib.request.urlopen(DATA + "course-chunks.json"))
sections = index["sections"]
questions = json.load(urllib.request.urlopen(DATA + "retrieval-questions.json"))["questions"]

docs = [f"{s['lesson']}: {s['section']}. {s['text']}" for s in sections]
target = [next(i for i, s in enumerate(sections) if s["page"] == q["page"] and s["section"].startswith(f"{q['section']}.")) for q in questions]

print(f"Loaded {len(docs)} course chunks and {len(questions)} test evaluation questions.")
print(f"Sample test: '{questions[0]['q']}' → target: '{sections[target[0]]['section']}'")

## 1. Building the document retriever index
Fit a `TfidfVectorizer` with `stop_words="english"` and `sublinear_tf=True` across all `docs`.
Compute the document matrix `D = tfidf.transform(docs)`.
Assign to `tfidf` and `D`.

In [ ]:
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...

In [ ]:
check(1, lambda: D.shape == (254, len(tfidf.get_feature_names_out())), "retriever index transforms 254 course chunks into normalized sparse TF-IDF vectors")

## 2. The top-k retriever
Write `retrieve(query, k=3)` that:
- Transforms `query` into vector `q_vec = tfidf.transform([query])`
- Computes cosine similarities: `scores = (q_vec @ D.T).toarray()[0]`
- Finds the top `k` chunk indices sorted descending by score: `np.argsort(-scores)[:k]`
Returns a list of the top `k` indices.

In [ ]:
def retrieve(query, k=3):
    # YOUR CODE HERE (stuck? hint(2), then answer(2))
    ...

In [ ]:
check(2, lambda: len(retrieve(questions[0]["q"], k=3)) == 3, "retrieve(query, k=3) returns list of top 3 chunk indices")

## 3. Measuring context recall on the eval set
Evaluate the retriever across all 35 questions.
A question is a "hit" if its ground-truth `target[i]` appears anywhere in the top 3 retrieved chunks.
Compute the fraction of hits across all 35 questions and store in `recall_at_3`.

In [ ]:
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...

if "recall_at_3" in locals() and recall_at_3 is not None:
    print(f"Retriever Context Recall@3: {recall_at_3 * 100:.1f}%")

In [ ]:
check(3, lambda: recall_at_3 > 0.70, "retriever successfully surfaces the ground-truth answer chunk in top-3 for over 70% of questions")

## 4. Formulating the grounded RAG context prompt
Write `build_rag_prompt(query, chunk_indices)` that assembles:
```text
CONTEXT:
[1] <chunk text 1>
[2] <chunk 2 text>
...
QUESTION: <query>
INSTRUCTIONS: Answer the question using ONLY the provided context. Cite sources using [1], [2].
```
Return the assembled string.

In [ ]:
def build_rag_prompt(query, chunk_indices):
    # YOUR CODE HERE (stuck? hint(4), then answer(4))
    ...

In [ ]:
check(4, lambda: "[1]" in build_rag_prompt(questions[0]["q"], [0, 1]) and "[2]" in build_rag_prompt(questions[0]["q"], [0, 1]) and questions[0]["q"] in build_rag_prompt(questions[0]["q"], [0, 1]), "RAG prompt builder attaches numbered source citations and user query")

## 5. Grounding evaluation: measuring faithfulness
When an LLM generates a response, we must verify that its assertions are grounded in the retrieved chunks,
rather than hallucinated facts or prior knowledge.

Write `grounding_ratio(answer, context_text)`:
- Extract lowercase alphabetic words of length $\ge 4$ from both `answer` and `context_text` using `re.findall(r'[a-z]{4,}', text.lower())`.
- If `answer_words` is empty, return 1.0.
- Compute the fraction of unique words in `answer_words` that appear in `context_words`:
  $$\text{ratio} = \frac{|\text{unique words in answer} \cap \text{unique words in context}|}{|\text{unique words in answer}|}$$
Return `ratio` as a float.

In [ ]:
def grounding_ratio(answer, context_text):
    # YOUR CODE HERE (stuck? hint(5), then answer(5))
    ...

In [ ]:
ctx = docs[target[0]]
grounded_ans = "The training error falls because high-degree polynomials bend to fit noisy points, but validation error rises [1]."
hallucinated_ans = "Quantum entanglement explains why deep quantum circuits outperform classical gradient descent on supercomputers."

check(5, lambda: grounding_ratio(grounded_ans, ctx) > 0.65 and grounding_ratio(hallucinated_ans, ctx) < 0.20, "grounding evaluator correctly gives high scores to faithful responses and flags hallucinations")

## 6. The complete QA Bot function
Write `qa_bot(query)` that:
1. Calls `indices = retrieve(query, k=2)`
2. Extracts top chunk: `best_doc = docs[indices[0]]`
3. Returns a dictionary:
   - `"query"`: query string
   - `"citations"`: `[sections[i]["lesson"] for i in indices]`
   - `"top_chunk"`: `best_doc`
   - `"is_grounded"`: `True`

In [ ]:
def qa_bot(query):
    # YOUR CODE HERE (stuck? hint(6), then answer(6))
    ...

In [ ]:
check(6, lambda: isinstance(qa_bot("Why does my model overfit on small datasets?"), dict) and len(qa_bot("Why does my model overfit on small datasets?")["citations"]) == 2, "QA bot retrieves relevant course chunks and returns verifiable citations")